# Fantasmas no Servico Publico

Este notebook cruza dados de **folha de pagamento federal** com registros de **vinculos empregatícios** (RAIS/CAGED) para identificar **servidores fantasmas** e inconsistencias na folha de pagamento do governo.

**Fontes utilizadas:**
- CGU / Portal da Transparencia — servidores federais e folha de pagamento
- Ministerio do Trabalho — RAIS (vinculos empregatícios anuais)
- Ministerio do Trabalho — CAGED (admissoes e desligamentos mensais)
- CGU — CEAF (servidores expulsos)

**Campos-ponte:** CPF, Nome, PIS/PASEP

**Dificuldade:** Intermediario

---

## 1. Setup e configuracao

Instale as dependencias com `pip install -r requirements.txt` antes de executar.

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path
from tabulate import tabulate

# ============================================================
# CONFIGURACAO
# ============================================================

# TODO: Insira sua chave de API do Portal da Transparencia
# Obtenha gratuitamente em: https://portaldatransparencia.gov.br/api-de-dados
API_KEY = "SEU_TOKEN_AQUI"

BASE_URL = "https://api.portaldatransparencia.gov.br/api-de-dados"
HEADERS = {"chave-api-dados": API_KEY, "Accept": "application/json"}

# TODO: Configure o orgao de interesse (ou deixe vazio para todos)
CODIGO_ORGAO = ""  # Ex: "26000" para Ministerio da Educacao
ANO_REFERENCIA = 2024

# Diretorio com microdados da RAIS
# Download: http://pdet.mte.gov.br/microdados-rais
DIRETORIO_RAIS = Path("./dados_rais")

# Diretorio com microdados do CAGED
# Download: http://pdet.mte.gov.br/novo-caged
DIRETORIO_CAGED = Path("./dados_caged")


def coletar_paginado(endpoint: str, params: dict, max_paginas: int = 20) -> list:
    """Coleta dados paginados de um endpoint da API CGU."""
    todos = []
    for pagina in range(1, max_paginas + 1):
        try:
            params["pagina"] = pagina
            resp = requests.get(
                f"{BASE_URL}/{endpoint}",
                headers=HEADERS,
                params=params,
                timeout=30,
            )
            resp.raise_for_status()
            dados = resp.json()
            if not dados:
                break
            todos.extend(dados)
            time.sleep(2)  # Respeitar rate limit (30 req/min)
        except Exception as e:
            print(f"Erro no endpoint {endpoint}, pagina {pagina}: {e}")
            break
    return todos


print("Setup concluido.")
print(f"Orgao: {CODIGO_ORGAO or 'Todos'}")
print(f"Ano de referencia: {ANO_REFERENCIA}")

## 2. Baixar folha de pagamento federal (Portal da Transparencia)

Coletar a lista de servidores federais ativos e suas remuneracoes.

**Endpoint:** `GET https://api.portaldatransparencia.gov.br/api-de-dados/servidores`  
**Params:** `codigoOrgaoExercicio`, `pagina`

**Alternativa (recomendada para volume):** Download em lote no [Portal da Transparencia](https://portaldatransparencia.gov.br/download-de-dados/servidores)

In [ ]:
# TODO: Coletar servidores federais
# Endpoint API: GET https://api.portaldatransparencia.gov.br/api-de-dados/servidores
# Params: codigoOrgaoExercicio (opcional), pagina
#
# Alternativa (download em lote, recomendado):
# https://portaldatransparencia.gov.br/download-de-dados/servidores
# Arquivos: {ANO}{MES}_Servidores_SIAPE.csv

# Via API:
# params_servidores = {}
# if CODIGO_ORGAO:
#     params_servidores["codigoOrgaoExercicio"] = CODIGO_ORGAO
# servidores_raw = coletar_paginado("servidores", params_servidores, max_paginas=100)
# df_servidores = pd.DataFrame(servidores_raw)

# Via download em lote (recomendado para grandes volumes):
# df_servidores = pd.read_csv(
#     f"./dados_servidores/{ANO_REFERENCIA}01_Servidores_SIAPE.csv",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

# TODO: Normalizar campos para cruzamento
# df_servidores["nome_norm"] = df_servidores["nome"].str.upper().str.strip()
# df_servidores["cpf_norm"] = df_servidores["cpf"].str.strip()

# TODO: Exibir resumo
# print(f"Servidores coletados: {len(df_servidores)}")
# print(f"Orgaos: {df_servidores['nomeOrgaoExercicio'].nunique()}")
# print(f"Remuneracao media: R$ {df_servidores['remuneracaoBasica'].astype(float).mean():,.2f}")

print("TODO: Coletar folha de pagamento federal")

## 3. Buscar vinculos empregatícios (RAIS / CAGED)

Carregar microdados da **RAIS** (Relacao Anual de Informacoes Sociais) para identificar todos os vinculos empregatícios formais registrados no ano.

**Fonte:** [PDET — Microdados RAIS](http://pdet.mte.gov.br/microdados-rais)  
**Fonte:** [PDET — Novo CAGED](http://pdet.mte.gov.br/novo-caged)

> **Nota:** Os microdados da RAIS sao volumosos (gigabytes). Filtre por UF ou atividade economica se necessario.

In [ ]:
# TODO: Carregar microdados da RAIS
# Download: http://pdet.mte.gov.br/microdados-rais
# Arquivo: RAIS_VINC_PUB_{UF}_{ANO}.txt (vinculos do setor publico)

# Campos relevantes da RAIS:
# - CPF (identificador do trabalhador - anonimizado em versoes publicas)
# - PIS (identificador alternativo)
# - Nome do trabalhador
# - CNPJ do empregador
# - Natureza juridica do empregador
# - Tipo de vinculo (CLT, estatutario, temporario)
# - Remuneracao media
# - Horas contratadas
# - Data de admissao e desligamento

# Exemplo de carregamento:
# df_rais = pd.read_csv(
#     DIRETORIO_RAIS / f"RAIS_VINC_PUB_SP_{ANO_REFERENCIA}.txt",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

# TODO: Normalizar campos para cruzamento
# df_rais["nome_norm"] = df_rais["Nome Trabalhador"].str.upper().str.strip()

# TODO: Carregar CAGED para dados mais recentes (mensal)
# Download: http://pdet.mte.gov.br/novo-caged
# df_caged = pd.read_csv(
#     DIRETORIO_CAGED / f"CAGEDMOV{ANO_REFERENCIA}01.txt",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

print("TODO: Carregar microdados da RAIS e/ou CAGED")

## 4. Cruzar CPFs entre fontes

Realizar o cruzamento principal: comparar a lista de servidores federais com os vinculos empregatícios da RAIS/CAGED.

**Objetivo:** Encontrar servidores que possuem **vinculos simultaneos** no setor privado, ou multiplos vinculos publicos incompativeis.

In [ ]:
# TODO: Cruzar servidores com vinculos da RAIS/CAGED

# Opcao 1: Cruzamento por CPF (quando disponivel)
# cruzamento_cpf = df_servidores.merge(
#     df_rais,
#     left_on="cpf_norm",
#     right_on="cpf_trabalhador",
#     how="inner",
#     suffixes=("_servidor", "_rais"),
# )

# Opcao 2: Cruzamento por nome (quando CPF nao e disponivel)
# cruzamento_nome = df_servidores.merge(
#     df_rais,
#     left_on="nome_norm",
#     right_on="nome_norm",
#     how="inner",
#     suffixes=("_servidor", "_rais"),
# )

# TODO: Filtrar vinculos relevantes
# Excluir vinculos no proprio setor publico federal (mesma fonte)
# Focar em:
# - Vinculos CLT em empresas privadas (natureza juridica != administracao publica)
# - Vinculos em tempo integral (>= 40h semanais)
# - Vinculos ativos (sem data de desligamento no ano)

# TODO: Identificar acumulacoes ilegais
# A Constituicao permite acumulacao apenas para:
# - Dois cargos de professor
# - Um cargo de professor + um tecnico/cientifico
# - Dois cargos de profissional de saude
# Qualquer outra combinacao e potencialmente ilegal

print("TODO: Realizar cruzamento entre servidores e vinculos RAIS/CAGED")

## 5. Identificar inconsistencias

Analisar os resultados do cruzamento para detectar padroes anomalos:

- **Multiplos vinculos em tempo integral** — servidor com 40h no setor publico e 40h no privado
- **Servidor em localidade incompativel** — lotado em Brasilia mas com vinculo CLT em outro estado
- **Servidor expulso na folha** — consta no CEAF (expulso) mas ainda recebe remuneracao
- **Vinculo pos-obito** — servidor falecido com pagamentos apos a data do obito

In [ ]:
# TODO: Identificar inconsistencias nos cruzamentos

# --- Tipo 1: Multiplos vinculos em tempo integral ---
# vinculos_duplos = cruzamento[
#     (cruzamento["horas_servidor"].astype(int) >= 40) &
#     (cruzamento["horas_rais"].astype(int) >= 40)
# ]

# --- Tipo 2: Servidor expulso ainda na folha ---
# Endpoint CEAF: GET https://api.portaldatransparencia.gov.br/api-de-dados/ceaf
# ceaf_raw = coletar_paginado("ceaf", params={}, max_paginas=50)
# df_ceaf = pd.DataFrame(ceaf_raw)
# if not df_ceaf.empty:
#     df_ceaf["nome_norm"] = df_ceaf["nomePunido"].str.upper().str.strip()
#     expulsos_na_folha = df_servidores[
#         df_servidores["nome_norm"].isin(df_ceaf["nome_norm"])
#     ]
#     if not expulsos_na_folha.empty:
#         print(f"[CRITICO] {len(expulsos_na_folha)} servidor(es) expulso(s) "
#               f"ainda constam na folha de pagamento!")

# --- Tipo 3: Vinculos em localidades incompativeis ---
# TODO: Comparar UF do orgao do servidor com UF do vinculo RAIS

# --- Tipo 4: Pagamentos pos-obito ---
# TODO: Cruzar com base de obitos (quando disponivel)
# Fonte possivel: SIM (Sistema de Informacoes sobre Mortalidade)

print("TODO: Analisar inconsistencias e classificar por tipo")

## 6. Gerar relatorio

Consolidar todas as inconsistencias encontradas em um relatorio final, com classificacao de gravidade e detalhamento de cada caso.

In [ ]:
# TODO: Gerar relatorio consolidado de inconsistencias

# inconsistencias = []

# TODO: Adicionar cada tipo de inconsistencia ao relatorio
# Para cada caso encontrado:
# inconsistencias.append({
#     "tipo": "Multiplos vinculos",  # ou "Servidor expulso", "Localidade incompativel"
#     "gravidade": "CRITICO",  # CRITICO, ALTO, MEDIO
#     "servidor": nome,
#     "orgao": orgao,
#     "cargo": cargo,
#     "remuneracao": remuneracao,
#     "detalhes": "Descricao da inconsistencia",
# })

# TODO: Exibir relatorio
# df_relatorio = pd.DataFrame(inconsistencias)
# if not df_relatorio.empty:
#     df_relatorio = df_relatorio.sort_values("gravidade")
#     print("=" * 60)
#     print("RELATORIO: FANTASMAS NO SERVICO PUBLICO")
#     print("=" * 60)
#     print(f"Total de inconsistencias: {len(df_relatorio)}")
#     print(f"  CRITICO: {len(df_relatorio[df_relatorio['gravidade'] == 'CRITICO'])}")
#     print(f"  ALTO: {len(df_relatorio[df_relatorio['gravidade'] == 'ALTO'])}")
#     print(f"  MEDIO: {len(df_relatorio[df_relatorio['gravidade'] == 'MEDIO'])}")
#     print()
#     print(tabulate(df_relatorio, headers="keys", tablefmt="grid"))
#     df_relatorio.to_csv("relatorio_fantasmas.csv", index=False)
#     print(f"\nRelatorio salvo em: relatorio_fantasmas.csv")
# else:
#     print("Nenhuma inconsistencia encontrada.")

print("TODO: Gerar relatorio consolidado")